## Crawilng and parsing API Documentation

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from typing import Set, List

BASE_URL = "https://wbsg-uni-mannheim.github.io/PyDI/"

def is_internal_link(url: str) -> bool:
    return url.startswith(BASE_URL)

def clean_text(soup: BeautifulSoup) -> str:
    # remove navigation, footer, sidebar
    for tag in soup(["nav", "footer", "header", "script", "style"]):
        tag.decompose()

    main = soup.find("div", class_="document")
    if not main:
        return ""

    return main.get_text(separator="\n", strip=True)


def crawl_pydi_docs(start_url: str) -> List[str]:
    visited: Set[str] = set()
    to_visit = [start_url]
    documents = []

    while to_visit:
        url = to_visit.pop()
        if url in visited:
            continue

        print(f"[*] Crawling: {url}")
        visited.add(url)

        try:
            resp = requests.get(url, timeout=10)
            soup = BeautifulSoup(resp.text, "html.parser")

            text = clean_text(soup)
            if text:
                documents.append(text)

            for link in soup.find_all("a", href=True):
                href = urljoin(url, link["href"])
                href = href.split("#")[0]  # remove anchors

                if is_internal_link(href) and href not in visited:
                    to_visit.append(href)

        except Exception as e:
            print(f"[x] Failed: {url} ({e})")

    return documents


In [2]:
docs = crawl_pydi_docs(BASE_URL)

[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/api/PyDI.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/api/PyDI.utils.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/PyDI/utils/similarity_registry.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/index.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/PyDI/utils/profiler.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/index.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_sources/index.rst.txt
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/PyDI/utils/llm.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/PyDI/utils/cluster_stats.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/_modules/PyDI/schemamatching/translator.html
[*] Crawling: https://wbsg-uni-mannheim.github.io/PyDI/api/PyDI.schemamatching.html
[*] Crawling: https://wbsg-uni

## Creating Vector DB

In [ ]:
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain_community.embeddings import SentenceTransformerEmbeddings
# from langchain.vectorstores import Chroma

# DB_DIR = "./pydi_apidocs_vector_db"


# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1200,
#     chunk_overlap=200,
# )

# chunks = []
# for doc in docs:
#     chunks.extend(splitter.split_text(doc))

# print(f"Total chunks: {len(chunks)}")

# embeddings = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# db = Chroma.from_texts(
#     texts=chunks,
#     embedding=embeddings,
#     persist_directory=DB_DIR
# )

# db.persist()
# print("[+] PyDI documentation vector DB built")


Total chunks: 1476


/var/folders/lx/mstfnlnx381664y7plq189hm0000gn/T/ipykernel_91402/3728183112.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


[+] PyDI documentation vector DB built


In [3]:
from dotenv import load_dotenv, find_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

load_dotenv(find_dotenv("../.env"))

DB_DIR = "./pydi_apidocs_vector_db"
BATCH_SIZE = 50 

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
)

chunks = []
for doc in docs:
    chunks.extend(splitter.split_text(doc))

print(f"Total chunks: {len(chunks)}")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

# Create empty Chroma DB
db = Chroma(
    embedding_function=embeddings,
    persist_directory=DB_DIR
)

# Batched ingestion
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    db.add_texts(batch)
    print(f"Embedded batch {i // BATCH_SIZE + 1}")

db.persist()
print("[+] PyDI documentation vector DB built successfully")


/Users/abd/Developer/team_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 1476


/var/folders/lx/mstfnlnx381664y7plq189hm0000gn/T/ipykernel_33801/2067529721.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(


Embedded batch 1
Embedded batch 2
Embedded batch 3
Embedded batch 4
Embedded batch 5
Embedded batch 6
Embedded batch 7
Embedded batch 8
Embedded batch 9
Embedded batch 10
Embedded batch 11
Embedded batch 12
Embedded batch 13
Embedded batch 14
Embedded batch 15
Embedded batch 16
Embedded batch 17
Embedded batch 18
Embedded batch 19
Embedded batch 20
Embedded batch 21
Embedded batch 22
Embedded batch 23
Embedded batch 24
Embedded batch 25
Embedded batch 26
Embedded batch 27
Embedded batch 28
Embedded batch 29
Embedded batch 30
[+] PyDI documentation vector DB built successfully


/var/folders/lx/mstfnlnx381664y7plq189hm0000gn/T/ipykernel_33801/2067529721.py:38: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [4]:
def load_reference_db():
    # embeddings = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

    return Chroma(
        persist_directory=DB_DIR,
        embedding_function=embeddings,
    )


def query_pydi_reference(query: str) -> str:
    db = load_reference_db()
    results = db.similarity_search(query, k=3)

    if not results:
        return "No relevant reference content found."

    output = "\n\n-----\n\n".join([r.page_content for r in results])
    return output

In [5]:
import textwrap

textwrap.wrap(query_pydi_reference('how to implement standard blocker?'))

['. logger . debug ( f "Creating candidate record pairs from { len (',
 'self . _common_keys ) } blocks" ) batch_pairs : List [ Tuple [ str ,',
 'str ]] = [] for key in self . _common_keys : left_ids = self .',
 '_left_blocks [ key ] right_ids = self . _right_blocks [ key ] #',
 'Cartesian within the block for lid in left_ids : for rid in right_ids',
 ': batch_pairs . append (( lid , rid )) if len ( batch_pairs ) >= self',
 '. batch_size : yield self . _emit_batch ( pd . DataFrame ( batch_pairs',
 ', columns = [ "id1" , "id2" ]) . assign ( block_key = key ) )',
 'batch_pairs = [] if batch_pairs : yield self . _emit_batch ( pd .',
 'DataFrame ( batch_pairs , columns = [ "id1" , "id2" ])) __all__ = [',
 '"StandardBlocker" ]  -----  with any other entity in the result.',
 'Parameters : threshold ( float ) preserve_scores ( bool )',
 'force_one_to_one ( bool ) cluster ( correspondences ) [source] \uf0c1 Apply',
 'stable matching to correspondences. Parameters : correspondences (',
 'Corres